# Practice: Are These Topics Worth Reporting?

**Computational Text Analysis Pack · Module 08 — Topic Modelling**

*Technique:* A topic model from k-selection to a defensible written interpretation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/05-unsupervised-structure/08e_practice_topics.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## The question

> **What themes run through pre-1929 European writing about Africa, and which of
> them can I defend in print?**

Modules 8a–8d gave LDA, NMF, coherence and the critique literature. This is the
workflow that produces a paragraph rather than a word cloud.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

docs = df_colonial["text"].astype(str).tolist()
cv = CountVectorizer(token_pattern=r"\w+", min_df=8, max_df=0.35, max_features=4000,
                     stop_words="english")
X = cv.fit_transform(docs)
vocab = np.array(cv.get_feature_names_out())
print(f"{X.shape[0]:,} passages, {X.shape[1]:,} vocabulary")

# Step 1: choose k by looking, not by a default.
binary = (X > 0).astype(int)
dfreq = np.asarray(binary.sum(axis=0)).ravel(); n = X.shape[0]
def coherence(model, k, top_n=10):
    scores = []
    for t in range(k):
        top = np.argsort(model.components_[t])[::-1][:top_n]
        for i in range(len(top)):
            for j in range(i+1, len(top)):
                co = int(binary[:, top[i]].multiply(binary[:, top[j]]).sum())
                if co == 0: scores.append(-1.0); continue
                p = co/n
                scores.append(np.log(p/((dfreq[top[i]]/n)*(dfreq[top[j]]/n)))/-np.log(p))
    return float(np.mean(scores))

rows = []
for k in [5, 8, 12, 20]:
    m = LatentDirichletAllocation(n_components=k, random_state=0, max_iter=15).fit(X)
    rows.append({"k": k, "coherence": coherence(m, k)})
    print(f"   k={k:>3}  coherence {rows[-1]['coherence']:.4f}")
print()
print("Chang et al: coherence can ANTI-correlate with interpretability. Use this")
print("to narrow the range, then read the topics before choosing.")

In [ ]:
K = 12
lda = LatentDirichletAllocation(n_components=K, random_state=0, max_iter=25).fit(X)
nmf = NMF(n_components=K, random_state=0, max_iter=400, init="nndsvda").fit(
    TfidfVectorizer(token_pattern=r"\w+", min_df=8, max_df=0.35,
                    max_features=4000, stop_words="english").fit_transform(docs))

print("LDA topics:")
for t in range(K):
    top = np.argsort(lda.components_[t])[::-1][:8]
    print(f"   {t:>2}: {' '.join(vocab[top])}")
print("\nNMF topics (same k, same corpus):")
for t in range(K):
    top = np.argsort(nmf.components_[t])[::-1][:8]
    print(f"   {t:>2}: {' '.join(vocab[top])}")
print()
print("Topics that appear in BOTH models are the ones to trust. A topic only one")
print("model finds is a property of the algorithm, not of the corpus.")

In [ ]:
# Step 3: ground each topic in documents, as Schmidt insists.
doc_topic = lda.transform(X)
TOPIC = 0        # <- change this
top_docs = np.argsort(doc_topic[:, TOPIC])[::-1][:3]
print(f"topic {TOPIC}: {' '.join(vocab[np.argsort(lda.components_[TOPIC])[::-1][:10]])}\n")
for i in top_docs:
    r = df_colonial.iloc[i]
    print(f"   [{doc_topic[i, TOPIC]:.2f}] {str(r['title'])[:46]} / {r['author']}")
    print("      " + " ".join(str(r["text"]).split())[:200] + "...\n")
print("If the passages do not read as the top words suggested, the topic is not")
print("what you were about to call it.")

In [ ]:
# Step 4: how much of the corpus does each topic actually account for?
share = doc_topic.mean(axis=0)
order = np.argsort(share)[::-1]
print(f"{'topic':>6}{'share':>9}   top words")
print("-" * 70)
for t in order:
    print(f"{t:>6}{share[t]:>9.3f}   {' '.join(vocab[np.argsort(lda.components_[t])[::-1][:6]])}")
plt.figure(figsize=(7, 3.6))
plt.bar(range(K), share[order], color="steelblue")
plt.xlabel("topic (sorted)"); plt.ylabel("mean share of corpus")
plt.title("Topic prevalence"); plt.tight_layout()

## What this licenses

**Defensible:** "An LDA model with k=12 over the corpus yields a topic
characterised by [words], which is also recovered by NMF at the same k, accounts
for X% of the corpus, and whose highest-weighted passages concern [what you
actually read]."

Every clause is checkable. Note the last one is the only one that required
reading, and it is the one carrying the interpretive claim.

**Not defensible:** naming a topic from its top ten words. That is the specific
practice Schmidt's *Words Alone* warns against, and module 8d measured how
little of a topic's mass those ten words hold.

**Also not defensible:** treating topic prevalence as a historical trend. This
corpus has no reliable dates and is dominated by a handful of authors — module
22a shows what that does to a time claim.

### 🔧 Your turn 1

Run the same procedure with k=5 and k=25. At which k do the topics you would actually write about appear?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Usually somewhere in the middle, and the useful discovery is that different k
values answer different questions: low k gives you the corpus's broad
divisions, high k gives you specific recurring scenes or vocabularies. Neither
is the "right" number. Reporting that you inspected several and chose one for a
stated reason is a stronger methods sentence than any coherence score, and it is
what the topic-modelling critique literature has been asking for since 2012.

</details>